In [ ]:
Q. 1 — Create a Table for Query Testing 
ANs:
CREATE DATABASE IF NOT EXISTS britmart_performance;
 
USE britmart_performance;
 
CREATE TABLE orders_perf (
    order_id INT PRIMARY KEY,
    customer_id INT NOT NULL,
    order_date DATE NOT NULL,
    city VARCHAR(50) NOT NULL,
    amount_gbp DECIMAL(10,2) NOT NULL,
    status VARCHAR(20) NOT NULL
);
 
INSERT INTO orders_perf VALUES
(1, 101, '2026-08-01', 'London', 45.50, 'complete'),
(2, 102, '2026-08-01', 'Manchester', 18.00, 'cancelled'),
(3, 101, '2026-08-02', 'London', 62.25, 'complete'),
(4, 103, '2026-08-02', 'Leeds', 30.00, 'complete'),
(5, 104, '2026-08-03', 'Bristol', 55.50, 'complete'),
(6, 102, '2026-08-03', 'Manchester', 12.00, 'pending'),
(7, 101, '2026-08-04', 'London', 80.00, 'complete'),
(8, 105, '2026-08-04', 'Leeds', 20.00, 'complete'),
(9, 106, '2026-08-05', 'Bristol', 100.00, 'complete'),
(10, 102, '2026-08-05', 'Manchester', 40.00, 'complete');

In [ ]:
Q. 2 — Inspect a Query Before Adding an Index
EXPLAIN
SELECT
    order_id,
    order_date,
    amount_gbp
FROM orders_perf
WHERE customer_id = 101;

In [ ]:
Q. 3 — Add an Index and Compare
Ans: 
CREATE INDEX idx_orders_customer
ON orders_perf(customer_id); 

EXPLAIN
SELECT
    order_id,
    order_date,
    amount_gbp
FROM orders_perf
WHERE customer_id = 101;

In [ ]:
Q. 4 — MCQ: Index
When is an index most likely to help? 
Ans: B. A query filters a large table using a selective column.

In [ ]:
Q. 5 — Compare SELECT * with Required Columns 
Explain why Query B is better when those are the only columns the consumer needs.
Ans:
Query B asks for only the required columns.

This can reduce unnecessary data transfer and makes the query intent clearer.

SELECT * is convenient during exploration, but it may read or return columns the consumer does not need.

In [ ]:
Q. 6 — Filter Before Returning Data 
Ans:
SELECT
    order_id,
    customer_id,
    order_date,
    amount_gbp
FROM orders_perf
WHERE status = 'complete'
  AND order_date >= '2026-08-03'
  AND amount_gbp >= 40
ORDER BY order_date, order_id;

In [ ]:
Q. 7 — Choose the Correct Join Key

In [ ]:
CREATE TABLE customers_perf (
    customer_id INT PRIMARY KEY,
    customer_name VARCHAR(100) NOT NULL,
    city VARCHAR(50) NOT NULL
);
 
INSERT INTO customers_perf VALUES
(101, 'Amelia Clarke', 'London'),
(102, 'Oliver Bennett', 'Manchester'),
(103, 'Aisha Khan', 'Leeds'),
(104, 'Harry Thompson', 'Bristol'),
(105, 'Emily Wilson', 'Leeds'),
(106, 'Noah Davies', 'Bristol');

In [ ]:
SELECT
    o.order_id,
    c.customer_name,
    o.amount_gbp,
    o.status
FROM orders_perf o
INNER JOIN customers_perf c
    ON o.customer_id = c.customer_id;

In [ ]:
customer_id identifies the customer relationship.
Joining on city would be unsafe because several customers can live in the same city, creating incorrect matches.

In [ ]:
Q. 8 — MCQ: Required Columns
Why should a query select only the columns it needs? 
Ans: C. It can reduce unnecessary data read or transferred.

In [ ]:
Q. 9 — Choose a Partitioning Column 
Ans: order_date is the strongest starting choice because the reports commonly filter by date.

A date-based layout can allow systems to skip data from dates that are not needed.

The exact partition design depends on data volume and query patterns.

In [ ]:
Q. 10 — Design a Date-Based Layout
Create a simple storage layout using the date.
Ans: 
orders/
├── order_date=2026-08-01/
├── order_date=2026-08-02/
└── order_date=2026-08-03/ 
A query for only 2026-08-02 can potentially avoid reading data from the other dates when the processing engine supports partition pruning.

In [ ]:
Q. 11 — Find the Small-Files Problem
Two designs contain the same amount of data. 
Ans:
Design B is more likely to create a small-files problem.

A very large number of tiny files can increase:

file listing work;
metadata operations;
task setup overhead.
Bigger files are not automatically better in every case, but millions of tiny files are usually inefficient for large analytical workloads.

In [ ]:
Q. 12 — MCQ: Compression
What is the main storage benefit of compression? 
Ans: A. It reduces the amount of storage required for compressible data. 

In [ ]:
Q. 13 — Compare the Formats 
Ans: 
Format	       Text or binary?	       Row or column oriented?	  Common strength
CSV            Text	                   Row-oriented	              Simple and widely supported
Parquet	       Binary	               Columnar	                  Analytical reads and compression
Avro	       Binary	               Row-oriented	              Record exchange with schema
ORC	           Binary	               Columnar	                  Analytical storage and compression


In [ ]:
Q. 14 — Choose a Format for Each Requirement
Choose CSV, Parquet, Avro, or ORC. 
Ans: 
A. A simple file that a business user can open easily. -> CSV
B. Analytical data where queries often read only a few columns. -> Parquet
C. Record-oriented data exchange where schema travels with the data. -> Avro
D. Columnar analytical storage used with systems that support ORC well. -> ORC 

There can be more than one valid format in real systems. The answer here is based on the main strengths covered in the lesson.

In [ ]:
Q. 15 — Inspect a CSV and List Its Limitations 
Ans: Possible observations:

CSV stores column names in the header, but it does not strongly store full data-type information.
The file is human-readable.
Values are stored as text.
CSV does not provide the same columnar read benefits as Parquet or ORC.
CSV is often larger than compressed binary formats for large analytical datasets.

In [ ]:
Q. 16 — MCQ: Parquet
Why is Parquet commonly used for analytical data?
Ans: D. Its columnar layout can help queries read only needed columns.

In [ ]:
Q. 17 — Build and Compare a Composite Index 
Ans:
DROP INDEX idx_orders_customer
ON orders_perf; 

EXPLAIN
SELECT
    order_id,
    order_date,
    amount_gbp
FROM orders_perf
WHERE customer_id = 101
  AND order_date >= '2026-08-02'; 

CREATE INDEX idx_orders_customer_date
ON orders_perf(customer_id, order_date); 

EXPLAIN
SELECT
    order_id,
    order_date,
    amount_gbp
FROM orders_perf
WHERE customer_id = 101
  AND order_date >= '2026-08-02';

In [ ]:
Q. 18 — Find an Index That May Not Help Much
Ans: 
Not necessarily.

The query needs to calculate a total across all order amounts and has no filter on customer_id.

The customer index does not directly reduce the number of rows needed for this result.

This is an example of why creating an index does not improve every query.

In [ ]:
Q. 19 — Review a Poor Storage Layout 
Ans: 
Possible improvements:

Organise data by a useful date partition.
Reduce the excessive number of tiny files.
Use a columnar analytical format such as Parquet or ORC when supported by the target system.
Use compression to reduce storage where appropriate.
Any three well-explained choices are acceptable.

In [ ]:
Q. 20 — Choose the Complete Design 
Ans: 
One sensible design is:
Partitioning:
By a useful date field because reports commonly filter by date.
 
File format:
Parquet or ORC because the workload is analytical and often reads only a few columns.
 
Compression:
Yes, because reducing stored data can reduce storage and I/O.
 
Query habit:
Filter early and select only the required columns.